# 02 · 학습 실행 및 모니터링
기본 `nnUNetTrainer`, `nnUNetPlans`, `3d_fullres`, fold 0부터 시작합니다.
기본 trainer는 1,000 epochs, epoch당 250 training iterations를 사용합니다. epoch는 데이터 전체 순회 횟수가 아닙니다.
기본 메모리 목표 8GB를 유지합니다. 24GB로 확대하지 않습니다.
전체 재현 시 config.server.json의 folds를 `[0,1,2,3,4]`로 변경하고 설정 셀을 다시 실행하세요.
같은 run_id에서 fold 목록만 확장할 수 있습니다. 학습 조건이 달라지면 새 run_id를 사용합니다.

In [ ]:
from pathlib import Path
import sys, os, json, shlex, subprocess
sys.dont_write_bytecode = True
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "nnunetv2").is_dir())
EXPERIMENT = REPO / "experiments/kits2023"
sys.path.insert(0, str(EXPERIMENT / "scripts"))
from workflow import read_config, paths, guard, environment, model_dir
CONFIG = EXPERIMENT / "config.server.json"
if not CONFIG.exists():
    CONFIG = EXPERIMENT / "config.json"  # 로컬 기본 설정: 실행 비활성화
cfg = read_config(CONFIG)
print("Config:", CONFIG)
print("Execution enabled:", cfg["execution_enabled"])
print("Storage:", cfg["storage_root"])


## 시작 또는 재개
준비 완료를 검사하고 tmux 세션으로 실행합니다. `RESUME=True`면 체크포인트에서 이어서 학습합니다.
이미 완료된 fold는 건너뜁니다. checkpoints가 없는 실패한 폴더는 자동 삭제하지 않습니다.
아래 실행 셀을 실행하면 실제 서버 학습이 시작됩니다.

In [ ]:
RESUME = False  # 중단한 실험을 이어갈 때만 True

guard(cfg)
p = paths(cfg)
assert (p["pre"]/cfg["dataset_name"]/"preparation.complete.json").exists(), "01_prepare 먼저 완료하세요"
session = "kits23-" + cfg["run_id"]
command = [sys.executable, "-u", str(EXPERIMENT/"scripts/run_folds.py"), "--config", str(CONFIG)]
if RESUME:
    command.append("--resume")
if subprocess.run(["tmux", "has-session", "-t", session], capture_output=True).returncode == 0:
    print("Already running:", session)
else:
    runner_log = p["root"] / (session + ".runner.log")
    subprocess.run(["tmux", "new-session", "-d", "-s", session,
                    shlex.join(command) + " >> " + shlex.quote(str(runner_log)) + " 2>&1"],
                   env=environment(cfg), check=True)
    print("Started:", session, "Runner log:", runner_log)


## Train loss + validation loss
동일 축에 두 곡선을 표시합니다. validation loss는 검증 패치 기반이며 전체 볼륨 Dice/NSD와 다릅니다.
손실에는 Dice 항이 포함되어 음수가 가능하며, train/validation의 augmentation 조건도 다릅니다.
`FOLD`를 바꾸거나 이 셀을 재실행하면 최신 로그를 읽습니다. 재개 시 같은 epoch의 최신 값을 사용합니다.

In [ ]:
guard(cfg)
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from workflow import parse_losses
FOLD = 0

def show_progress(fold):
    folder = model_dir(cfg) / f"fold_{fold}"
    df = pd.DataFrame(parse_losses(folder))
    if df.empty:
        print("아직 완료된 epoch 로그가 없습니다:", folder)
        return
    ax = df.plot(x="epoch", y=["train_loss", "val_loss"], figsize=(11, 4), grid=True)
    ax.set(title=f"KiTS 2023 / fold {fold}", ylabel="Loss")
    plt.show()
    display(df.tail(5))
    if "epoch_seconds" in df:
        mean_seconds = df.epoch_seconds.dropna().tail(10).mean()
        remaining_epochs = max(0, 1000 - int(df.epoch.max()) - 1)
        print("Estimated remaining training hours (this fold, excluding final validation):",
              round(remaining_epochs * mean_seconds / 3600, 2))
    status = paths(cfg)["run"] / f"fold_{fold}.status.json"
    if status.exists():
        display(json.loads(status.read_text()))
    gpu_path = paths(cfg)["run"] / "logs" / f"fold_{fold}.gpu.csv"
    if gpu_path.exists() and gpu_path.stat().st_size:
        gpu = pd.read_csv(gpu_path, names=["timestamp", "memory_MiB", "utilization_percent"])
        print("Sampled peak GPU memory (device-wide):", gpu.memory_MiB.max(), "MiB")
    return df

losses = show_progress(FOLD)


## 선택: 1분 동안 자동 갱신
4회만 갱신합니다. 이 셀을 중단해도 tmux 학습은 유지됩니다.

In [ ]:
AUTO_REFRESH = False
if AUTO_REFRESH:
    import time
    from IPython.display import clear_output
    for _ in range(4):
        clear_output(wait=True)
        show_progress(FOLD)
        time.sleep(15)


## 오류 및 최근 출력 확인
최종 체크포인트와 validation/summary.json이 모두 생성되어야 완료로 처리합니다. 중단 시 최근 저장 체크포인트 이후의 epoch는 다시 수행될 수 있습니다.

In [ ]:
from collections import deque
p = paths(cfg)
for fold in cfg["folds"]:
    status = p["run"] / f"fold_{fold}.status.json"
    print(f"Fold {fold}:", json.loads(status.read_text()) if status.exists() else "not started")
log = p["run"] / "logs" / f"fold_{FOLD}.console.log"
if log.exists():
    with log.open(errors="replace") as f:
        print("".join(deque(f, maxlen=30)))
